[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sk-2103/AI4Sustainability-Labs/blob/main/labs/lab05_lstm_streamflow/lab05_evaluation.ipynb)  
**AI for Sustainability** · University of Wisconsin–Madison · Labs by Saurabh Kaushik · Course instructor: Dr. Beth Tellman  
*Make a copy (File → Save a copy in Drive) before you start. Part of the [AI for Sustainability lab series](https://github.com/Sk-2103/AI4Sustainability-Labs).*

# Lab 05 — Evaluation Notebook
## LSTM for Streamflow Discharge Forecasting
### AI for Sustainability | University of Wisconsin–Madison

---

> **Complete this notebook AFTER finishing Lab 05.**  
> All answers must be written in the designated cells below.  
> Run all code cells and ensure outputs are visible before submitting.

---

## 📋 Grading Breakdown

| Section | Type | Marks |
|---------|------|-------|
| **Part A — Theory Questions** | Written answers (Markdown) | **13 marks** |
| **Part B — Practical Tasks** | Code + written interpretation | **12 marks** |
| **Total** | | **25 marks** |

---

## 📌 Submission Instructions

1. Complete **every** question in order  
2. Run all code cells so outputs are visible  
3. Write theory answers **in the Markdown cells provided** (replace the `*Your answer here*` placeholder)  
4. Download notebook, make changes and Submit the `.ipynb` file with your **UW NetID** in the filename: `lab05_eval_NETID.ipynb`

---

## ⚙️ Setup — Run This First

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SETUP — run this cell first, it is fully self-contained
# No Lab 05 session needed. Everything is defined here.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── 0. Install dependencies ───────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'dataretrieval', '--quiet'], check=False)

# ── 1. Imports ────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── 2. Dataset class ──────────────────────────────────────────────
class HydrologyDataset(Dataset):
    """Sliding-window dataset: each sample = (seq_len days of features, next-day discharge)."""
    def __init__(self, X, y, seq_len):
        self.X, self.y, self.seq_len = X, y, seq_len
    def __len__(self):
        return len(self.X) - self.seq_len
    def __getitem__(self, idx):
        return self.X[idx : idx + self.seq_len], self.y[idx + self.seq_len]

# ── 3. LSTM model ─────────────────────────────────────────────────
class LSTMForecaster(nn.Module):
    """Single- or multi-layer LSTM with a linear readout head."""
    def __init__(self, input_size, hidden_size=128, num_layers=1,
                 dropout=0.4, initial_forget_bias=3):
        super().__init__()
        self.lstm   = nn.LSTM(input_size, hidden_size, num_layers,
                              batch_first=True,
                              dropout=dropout if num_layers > 1 else 0.0)
        self.drop   = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_size, 1)
        # Forget-bias trick: initialise gates open so early training retains memory
        for name, p in self.lstm.named_parameters():
            if 'bias' in name:
                n = p.shape[0]
                nn.init.constant_(p[n // 4 : n // 2], initial_forget_bias)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(self.drop(out[:, -1, :]))

# ── 4. Training function ──────────────────────────────────────────
def train_model(train_dl, val_dl, hidden_size=128, num_layers=1,
                dropout=0.4, lr=1e-3, n_epochs=60, patience=10):
    """Train LSTMForecaster; returns (best_model, train_losses, val_losses)."""
    model = LSTMForecaster(
        input_size  = 4,          # prcp, tmean, tmax, swe
        hidden_size = hidden_size,
        num_layers  = num_layers,
        dropout     = dropout,
        initial_forget_bias = 3
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5, min_lr=1e-6)
    criterion = nn.MSELoss()

    best_val, best_state, pat = float('inf'), None, 0
    train_losses, val_losses  = [], []

    for epoch in range(1, n_epochs + 1):
        # Train
        model.train()
        tl = 0.0
        for Xb, yb in train_dl:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tl += loss.item()
        tl /= len(train_dl)

        # Validate
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for Xb, yb in val_dl:
                vl += criterion(model(Xb.to(device)), yb.to(device)).item()
        vl /= len(val_dl)

        train_losses.append(tl)
        val_losses.append(vl)
        scheduler.step(vl)

        if vl < best_val:
            best_val   = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
        if pat >= patience:
            print(f'   Early stop at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return model, train_losses, val_losses

# ── 5. Inference & metrics ────────────────────────────────────────
def run_inference(model, loader, target_scaler, device):
    """Run model on a DataLoader; return (predictions, observations) in m³/s."""
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            preds.append(model(Xb.to(device)).cpu().numpy())
            trues.append(yb.numpy())
    preds = target_scaler.inverse_transform(
        np.concatenate(preds).reshape(-1, 1)).flatten()
    trues = target_scaler.inverse_transform(
        np.concatenate(trues).reshape(-1, 1)).flatten()
    return preds, trues

def compute_metrics(obs, sim):
    """Return dict of NSE, KGE, PBIAS, RMSE, R."""
    obs, sim = np.array(obs, dtype=float), np.array(sim, dtype=float)
    nse   = 1 - np.sum((obs - sim)**2) / np.sum((obs - obs.mean())**2)
    r     = np.corrcoef(obs, sim)[0, 1]
    beta  = sim.mean() / obs.mean()
    gamma = (sim.std()  / sim.mean()) / (obs.std() / obs.mean())
    kge   = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)
    pbias = 100 * (sim - obs).sum() / obs.sum()
    rmse  = np.sqrt(np.mean((obs - sim)**2))
    return dict(NSE=round(nse,4), KGE=round(kge,4),
                PBIAS=round(pbias,2), RMSE=round(rmse,3), R=round(r,4))

# ── 6. Data download for a given USGS gauge ───────────────────────
def download_gauge(site_id, start='2000-01-01', end='2019-12-31'):
    """
    Download USGS discharge + DAYMET forcing for any gauge.
    Returns a daily DataFrame with columns:
        discharge, prcp, tmean, tmax, tmin, swe, dayl
    Raises RuntimeError with a clear message on any failure.
    """
    from dataretrieval import nwis
    import requests
    from io import StringIO

    # ── Discharge ─────────────────────────────────────────────────
    print(f'📡 USGS discharge  {site_id} ...')
    dv, _ = nwis.get_dv(sites=site_id, start=start, end=end, parameterCd='00060')
    if dv.empty:
        raise RuntimeError(
            f'No discharge data for site {site_id}.\n'
            f'Check: https://waterdata.usgs.gov/monitoring-location/{site_id}/')
    dv.index = pd.to_datetime(dv.index).tz_localize(None)
    col = [c for c in dv.columns if '00060' in c and 'Mean' in c]
    if not col:
        raise RuntimeError(f'Discharge column not found. Got: {list(dv.columns)}')
    dv['discharge'] = dv[col[0]] * 0.028317   # cfs → m³/s
    dv = dv[['discharge']].copy()
    print(f'   ✅ {len(dv)} records')

    # ── Lat / lon via USGS REST ───────────────────────────────────
    r_info = requests.get(
        f'https://waterservices.usgs.gov/nwis/site/'
        f'?format=rdb&sites={site_id}&siteOutput=Expanded', timeout=30)
    r_info.raise_for_status()
    lines = [l for l in r_info.text.splitlines()
             if not l.startswith('#') and l.strip()]
    info  = dict(zip(lines[0].split('\t'), lines[2].split('\t')))
    lat, lon = float(info['dec_lat_va']), float(info['dec_long_va'])
    print(f'   📍 lat={lat:.4f}  lon={lon:.4f}')

    # ── DAYMET forcing ────────────────────────────────────────────
    print(f'📡 DAYMET forcing ...')
    r = requests.get(
        f'https://daymet.ornl.gov/single-pixel/api/data'
        f'?lat={lat}&lon={lon}&vars=prcp,tmax,tmin,swe,dayl'
        f'&start={start}&end={end}&format=csv', timeout=60)
    r.raise_for_status()

    # Find header row dynamically (robust to DAYMET format changes)
    lines_d = r.text.splitlines()
    hdr_idx = next((i for i, l in enumerate(lines_d)
                    if 'year' in l.lower() and 'yday' in l.lower()), None)
    if hdr_idx is None:
        raise RuntimeError('DAYMET header not found. First 10 lines:\n' +
                           '\n'.join(lines_d[:10]))
    forcing = pd.read_csv(StringIO(r.text), skiprows=hdr_idx)
    forcing.columns = forcing.columns.str.strip()
    forcing['date'] = pd.to_datetime(
        forcing['year'].astype(str) + '-' +
        forcing['yday'].astype(str).str.zfill(3), format='%Y-%j')
    forcing = forcing.set_index('date')
    col_map = {c: ('prcp'  if 'prcp'  in c.lower() else
                   'tmax'  if 'tmax'  in c.lower() else
                   'tmin'  if 'tmin'  in c.lower() else
                   'swe'   if 'swe'   in c.lower() else
                   'dayl'  if 'dayl'  in c.lower() else c)
               for c in forcing.columns}
    forcing = forcing.rename(columns=col_map)
    forcing = forcing[['prcp','tmax','tmin','swe','dayl']].copy()
    forcing['tmean'] = (forcing['tmax'] + forcing['tmin']) / 2
    print(f'   ✅ {len(forcing)} records')

    # ── Merge ─────────────────────────────────────────────────────
    df = dv.join(forcing, how='inner').dropna()
    if len(df) < 365:
        raise RuntimeError(f'Only {len(df)} days after merge — too short.')
    print(f'\n🟢 {site_id} ready: {len(df)} days  '
          f'({df.index[0].date()} → {df.index[-1].date()})')
    return df

# ── 7. Download Wisconsin River (baseline gauge for Q6) ───────────
print('━'*60)
print('Downloading Wisconsin River data for Q6 baseline ...')
print('━'*60)
df = download_gauge('05397500')

print(f'\n✅ Setup complete')
print(f'   Device  : {device}')
print(f'   Dataset : {df.shape[0]} days × {df.shape[1]} variables')
print(f'   Columns : {list(df.columns)}')
df.head(3)


---
# Part A — Theory Questions
### Total: 13 marks

Answer in the Markdown cells provided. Be concise and precise — bullet points are fine where appropriate.

---

## Q1 — Why Sequential Data Needs Special Architectures [3 marks]

> 📖 *Refer to: **Lab 05 Part 1** — "Why Sequence Matters in Hydrology" (markdown cells + Figure 1)*  
> 📚 *Further reading: [Colah's LSTM Guide](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) — Sections 1 & 2*

**(a) [1 mark]**  
A standard feedforward ANN is given today's rainfall and temperature as inputs and asked to predict today's river discharge.  
Explain **in one sentence** why this ANN will fail for a snowmelt-dominated river like the Wisconsin River at Merrill, WI.

**(b) [1 mark]**  
The table below from Lab 05 Part 1.2 compares network architectures.  
Fill in the missing entry by selecting the correct option:

| Network | Memory | Suitable for long-range hydrology? |
|---------|--------|-----------------------------------|
| Feedforward ANN | None | ❌ |
| RNN | Short | ⚠️ |
| **LSTM** | **???** | **???** |

What are the two missing values? Justify in one sentence.

**(c) [1 mark]**  
The vanishing gradient equation from Lab 05 Part 1.3 is:

$$\frac{\partial L}{\partial W} = \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

If each term equals 0.9, after how many steps does the gradient drop below 0.01?  
Show your calculation. What does this mean for predicting spring snowmelt from January snowpack?

---
*Your answers (Q1):*

**(a)**

**(b)**

**(c)**


## Q2 — The LSTM Cell: Gates and Memory [3 marks]

> 📖 *Refer to: **Lab 05 Part 4** — "LSTM Architecture Deep Dive" (markdown + Figure 4)*  
> 📚 *Further reading: [Colah's LSTM Guide](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) — "The Core Idea Behind LSTMs" section*

An LSTM cell has three gates and two states. The cell state update equation is:

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

**(a) [1 mark]**  
Match each gate to its role by completing the table:

| Gate | Symbol | Role (choose from list below) |
|------|--------|-------------------------------|
| Forget gate | $f_t$ | ??? |
| Input gate | $i_t$ | ??? |
| Output gate | $o_t$ | ??? |

**Roles:** "Decides what to expose as the hidden state output" / "Decides what old memory to erase" / "Decides what new information to write into memory"

**(b) [1 mark]**  
In Lab 05, the model is initialized with `initial_forget_bias = 3`.  
Given that $\sigma(3) \approx 0.95$, explain what the forget gate does at the **start of training** and why this benefits a snowmelt-dominated catchment.

**(c) [1 mark]**  
The Lab 05 Part 4 hydrology analogy table maps LSTM components to Wisconsin River processes.  
Complete the following analogy in your own words (do not just copy from the notebook):

- Cell state $c_t$ ↔ *[your answer]*  
- Forget gate $f_t$ ↔ *[your answer]*  
- Output gate $o_t$ ↔ *[your answer]*

---
*Your answers (Q2):*

**(a)**

**(b)**

**(c)**


## Q3 — Data Preparation & the Sliding Window [3 marks]

> 📖 *Refer to: **Lab 05 Part 3** — "Data Preprocessing & Sliding Window" (markdown + Figure 3 + preprocessing code cell)*  
> 📚 *Further reading: [Why random splits leak the future](https://towardsdatascience.com/time-series-machine-learning-regression-framework-9ea33929009a)*

**(a) [1 mark]**  
Lab 05 uses a **chronological** train/val/test split (2000–2013 / 2014–2016 / 2017–2019) instead of a random 80/20 split.  
What is **data leakage** in the context of time series, and why would a random split cause it here?

**(b) [1 mark]**  
The preprocessing cell fits the `MinMaxScaler` **only on the training set**:

```python
feat_scaler = MinMaxScaler()
X_train = feat_scaler.fit_transform(df_train[FEATURES])
X_val   = feat_scaler.transform(df_val[FEATURES])   # note: transform only, not fit_transform
```

Why is it wrong to call `fit_transform` on the validation or test set?  
What information would leak if you did?

**(c) [1 mark]**  
In Figure 3, the sliding window is shown with `SEQ_LEN_DEMO = 7` for illustration, but the actual model uses `SEQ_LEN = 365`.  
If the training set has **5,110 days** and `SEQ_LEN = 365`, how many training samples does `HydrologyDataset.__len__()` return?  
Show your calculation and explain what each sample represents physically.

---
*Your answers (Q3):*

**(a)**

**(b)**

**(c)**


## Q4 — Training Dynamics [2 marks]

> 📖 *Refer to: **Lab 05 Part 5** — "Training" (training loop cell + Figure 5)*  
> 📚 *Further reading: [Adam optimizer paper (Kingma & Ba 2015)](https://arxiv.org/abs/1412.6980)*

**(a) [1 mark]**  
Look at the training loop in Lab 05 Part 5. These five lines appear inside every batch iteration:

```python
optimizer.zero_grad()
pred = model(Xb)
loss = criterion(pred, yb)
loss.backward()
optimizer.step()
```

Briefly describe what **each line does** (one phrase per line). You do not need to explain the math — focus on the purpose.

**(b) [1 mark]**  
Figure 5 shows training and validation loss curves on a **log scale**.  
Describe what the **gap between training and validation loss** indicates, and explain how `patience=10` early stopping addresses it. What model checkpoint is restored at the end of training, and why?

---
*Your answers (Q4):*

**(a)**

**(b)**


## Q5 — Evaluation Metrics [2 marks]

> 📖 *Refer to: **Lab 05 Part 6** — "Evaluation & Visualization" (markdown + evaluation code cell)*  
> 📚 *Further reading: [Moriasi et al. 2007 — model evaluation guidelines](https://doi.org/10.13031/2013.23153)*

The NSE formula from Lab 05 Part 6 is:

$$\text{NSE} = 1 - \frac{\sum(Q_{obs} - Q_{sim})^2}{\sum(Q_{obs} - \bar{Q}_{obs})^2}$$

**(a) [1 mark]**  
What does NSE = 0 mean physically? What "model" does NSE = 0 correspond to?  
Why is NSE preferred over plain RMSE for comparing models across catchments of different sizes?

**(b) [1 mark]**  
Lab 05 Part 6 evaluation code computes three metrics: NSE, KGE, and PBIAS.  
The KGE decomposes into three terms: $r$ (correlation), $\beta$ (bias ratio), and $\gamma$ (variability ratio).  
If a model has **high NSE but KGE much lower than NSE**, which component of KGE is most likely causing the discrepancy — and what type of model error does it point to? (Hint: think about what bias or variability errors look like on a scatter plot.)

---
*Your answers (Q5):*

**(a)**

**(b)**


---
# Part B — Practical Tasks
### Total: 12 marks

Run all code cells. Your outputs (plots, printed metrics) **must be visible** in the submitted notebook.

> ⚠️ **Academic integrity:** The scaffolding below provides the full pipeline. You are asked to fill specific `???` gaps and write your own observations. Submissions that are verbatim copies of Lab 05 will receive zero marks for the relevant task.

---


## Q6 — LSTM Architecture Experiment  [6 marks]

> 📖 *Refer to: **Lab 05 Part 4** (model definition) and **Part 5** (training loop)*

The baseline LSTM uses `hidden_size=128`, `num_layers=1`, `dropout=0.4`, `SEQ_LEN=365`.  
You will **modify two hyperparameters**, retrain, and interpret what changes — using the Wisconsin River data already downloaded above.

---
### Q6a — Prepare Data & Train Baseline  [1 mark]

This cell is **complete — just run it.** Record the printed NSE/KGE/PBIAS — you will need them for Q6c.


In [ ]:
# ── Q6a: Prepare data + train baseline — COMPLETE, just run ────────────────

FEATURES   = ['prcp', 'tmean', 'tmax', 'swe']
TARGET     = 'discharge'
SEQ_LEN    = 7
BATCH_SIZE = 128

# Chronological split
df_train = df['2000-01-01':'2013-12-31']
df_val   = df['2014-01-01':'2016-12-31']
df_test  = df['2017-01-01':'2019-12-31']

# Fit scalers on training only — never on val/test
feat_scaler   = MinMaxScaler()
target_scaler = MinMaxScaler()
X_train = torch.FloatTensor(feat_scaler.fit_transform(df_train[FEATURES]))
X_val   = torch.FloatTensor(feat_scaler.transform(df_val[FEATURES]))
X_test  = torch.FloatTensor(feat_scaler.transform(df_test[FEATURES]))
y_train = torch.FloatTensor(target_scaler.fit_transform(df_train[[TARGET]]))
y_val   = torch.FloatTensor(target_scaler.transform(df_val[[TARGET]]))
y_test  = torch.FloatTensor(target_scaler.transform(df_test[[TARGET]]))

train_dl = DataLoader(HydrologyDataset(X_train, y_train, SEQ_LEN),
                      batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_dl   = DataLoader(HydrologyDataset(X_val,   y_val,   SEQ_LEN),
                      batch_size=BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(HydrologyDataset(X_test,  y_test,  SEQ_LEN),
                      batch_size=BATCH_SIZE, shuffle=False)

# Train baseline
print('🚀 Training baseline  (hidden=128, layers=1, dropout=0.4) ...')
baseline_model, bl_tr_loss, bl_va_loss = train_model(
    train_dl, val_dl, hidden_size=128, num_layers=1, dropout=0.4)

bl_preds, bl_obs = run_inference(baseline_model, test_dl, target_scaler, device)
bl_m = compute_metrics(bl_obs, bl_preds)
print(f'\n📊 Baseline results')
print(f'   NSE   = {bl_m["NSE"]}')
print(f'   KGE   = {bl_m["KGE"]}')
print(f'   PBIAS = {bl_m["PBIAS"]} %')


---
### Q6b — Modify the Architecture and Retrain  [3 marks]

Choose **any two** changes from the table and fill in the `???` values:

| Option | Parameter | Baseline → New value |
|--------|-----------|----------------------|
| A | `hidden_size` | 128 → **256** (larger) |
| B | `hidden_size` | 128 → **64** (smaller) |
| C | `num_layers`  | 1 → **2** (deeper) |
| D | `dropout`     | 0.4 → **0.2** (less regularisation) |
| E | `dropout`     | 0.4 → **0.6** (more regularisation) |

**Your chosen changes** *(fill in before running)*:  
- Change 1: `???` from `???` → `???`  
- Change 2: `???` from `???` → `???`


In [ ]:
# ── Q6b: Modified model — fill in the three ??? values then run ─────────────

print('🚀 Training modified model ...')
modified_model, mod_tr_loss, mod_va_loss = train_model(
    train_dl, val_dl,
    hidden_size = ???,    # ← change 1 (or leave 128 if not changing this)
    num_layers  = ???,    # ← change 2 (or leave 1 if not changing this)
    dropout     = ???,    # ← change 3 (or leave 0.4 if not changing this)
)

mod_preds, mod_obs = run_inference(modified_model, test_dl, target_scaler, device)
mod_m = compute_metrics(mod_obs, mod_preds)
print(f'\n📊 Modified model results')
print(f'   NSE   = {mod_m["NSE"]}')
print(f'   KGE   = {mod_m["KGE"]}')
print(f'   PBIAS = {mod_m["PBIAS"]} %')


**Plot training curves** — fill in the `???` description of your modified model:


In [ ]:
# ── Q6b: Loss curves — fill in the ??? description then run ────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
configs = [
    ('Baseline  (hidden=128, layers=1, dropout=0.4)', bl_tr_loss,  bl_va_loss),
    ('Modified  (???)',                               mod_tr_loss, mod_va_loss),  # ← describe your changes
]
for ax, (title, tl, vl) in zip(axes, configs):
    ax.semilogy(tl, label='Train loss', color='#2980b9', lw=2)
    ax.semilogy(vl, label='Val loss',   color='#e74c3c', lw=2, linestyle='--')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss (log scale)')
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Q6b: Training Dynamics — Baseline vs Modified', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
### Q6c — Compare and Interpret  [2 marks]

First fill in the table with your actual numbers, then answer all three questions.

| | Baseline | Modified model |
|--|----------|----------------|
| `hidden_size` | 128 | |
| `num_layers` | 1 | |
| `dropout` | 0.4 | |
| **Test NSE** | | |
| **Test KGE** | | |
| **PBIAS (%)** | | |
| Epochs trained | | |

---

**(i)** Did NSE improve or worsen after your changes? Give a reason grounded in what those specific hyperparameter changes do mathematically or physically — do not just say "because I changed the architecture."

*Your answer:*

**(ii)** Look at your two loss curve plots. Did the train–validation gap increase or decrease in the modified model? What does this tell you about overfitting?

*Your answer:*

**(iii)** If you were deploying this model operationally for flood early warning, which model would you choose and why? Consider accuracy (NSE), reliability (KGE), and computational cost.

*Your answer:*


---
## Q7 — Apply the Model to a New Catchment  [6 marks]

> 📖 *Refer to: **Lab 05 Part 2** (data download) and the `download_gauge()` function defined in the Setup cell above*

The Wisconsin River is snowmelt-dominated and needs 365-day memory. Now apply the **same pipeline** to a contrasting catchment and explain the differences.

**Choose ONE gauge** and write its ID in Q7a:

| Gauge ID | Location | Hydrology |
|----------|----------|-----------|
| `05543500` | Fox River at Algonquin, IL | Flat, agricultural — flashy rapid runoff |
| `04085427` | Manitowoc River, WI | Small, rain-dominated — weak snowmelt |
| `05430500` | Yahara River, WI | Urban, lake-buffered — very smooth signal |
| `05414000` | Galena River, IL | Karst, groundwater-dominated — long memory |

**Your chosen gauge:** *(write here)*  
**Why you chose it:** *(one sentence — what hydrological contrast interests you?)*

---
### Q7a — Download Data  [1 mark]

Fill in the one `???` with your chosen gauge ID, then run.


In [ ]:
# ── Q7a: Download new gauge — fill in the ??? then run ──────────────────────

NEW_SITE_ID = '???'    # ← replace with your chosen gauge ID (keep the quotes)

df_new = download_gauge(NEW_SITE_ID)

print(f'\n📊 Summary — USGS {NEW_SITE_ID}')
print(df_new[['discharge','prcp','tmean','swe']].describe().round(2))
print(f'\nDischarge  mean={df_new.discharge.mean():.1f}  '
      f'median={df_new.discharge.median():.1f}  '
      f'max={df_new.discharge.max():.1f}  '
      f'min={df_new.discharge.min():.1f}  m³/s')


---
### Q7b — Preprocess and Train  [2 marks]

This cell is **complete — just run it.** It trains the same baseline LSTM on your new gauge.


In [ ]:
# ── Q7b: Preprocess + train on new gauge — COMPLETE, just run ───────────────

FEATURES = ['prcp', 'tmean', 'tmax', 'swe']
TARGET   = 'discharge'
SEQ_LEN  = 7

# Chronological split
df_new_train = df_new['2000-01-01':'2013-12-31']
df_new_val   = df_new['2014-01-01':'2016-12-31']
df_new_test  = df_new['2017-01-01':'2019-12-31']

# Fit scalers on training only
fs_new = MinMaxScaler()
ts_new = MinMaxScaler()
Xtr = torch.FloatTensor(fs_new.fit_transform(df_new_train[FEATURES]))
Xva = torch.FloatTensor(fs_new.transform(df_new_val[FEATURES]))
Xte = torch.FloatTensor(fs_new.transform(df_new_test[FEATURES]))
ytr = torch.FloatTensor(ts_new.fit_transform(df_new_train[[TARGET]]))
yva = torch.FloatTensor(ts_new.transform(df_new_val[[TARGET]]))
yte = torch.FloatTensor(ts_new.transform(df_new_test[[TARGET]]))

new_tr_dl = DataLoader(HydrologyDataset(Xtr, ytr, SEQ_LEN), batch_size=128, shuffle=True,  drop_last=True)
new_va_dl = DataLoader(HydrologyDataset(Xva, yva, SEQ_LEN), batch_size=128, shuffle=False)
new_te_dl = DataLoader(HydrologyDataset(Xte, yte, SEQ_LEN), batch_size=128, shuffle=False)

print(f'🚀 Training on USGS {NEW_SITE_ID} ...')
new_model, new_tr_loss, new_va_loss = train_model(new_tr_dl, new_va_dl)

new_preds, new_obs = run_inference(new_model, new_te_dl, ts_new, device)
new_m = compute_metrics(new_obs, new_preds)
print(f'\n📊 Results — USGS {NEW_SITE_ID}')
print(f'   NSE   = {new_m["NSE"]}')
print(f'   KGE   = {new_m["KGE"]}')
print(f'   PBIAS = {new_m["PBIAS"]} %')


---
### Q7c — Plot Test Hydrograph  [1 mark]

Fill in the two `???` array names, then run.


In [ ]:
# ── Q7c: Test hydrograph — fill in the two ??? array names then run ─────────

test_dates_new = df_new_test.index[SEQ_LEN:]

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(test_dates_new, ???,              # ← observed array  (hint: new_obs)
        label='Observed', color='#1a6ea8', lw=1.5, alpha=0.9)
ax.plot(test_dates_new, ???,              # ← predicted array (hint: new_preds)
        label='LSTM Predicted', color='#e74c3c', lw=1.5, linestyle='--')
ax.fill_between(test_dates_new, new_obs, new_preds,
                alpha=0.15, color='#e74c3c', label='Error')
ax.set_title(
    f'Q7c: Test Hydrograph — USGS {NEW_SITE_ID}  |  '
    f'NSE={new_m["NSE"]}  KGE={new_m["KGE"]}  PBIAS={new_m["PBIAS"]}%',
    fontsize=11, fontweight='bold')
ax.set_ylabel('Discharge (m³/s)', fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()


---
### Q7d — Written Comparison  [2 marks]

Fill in the right-hand column of the table with your actual values, then answer the three questions below.  
**Every answer must reference specific numbers from your outputs — no generic statements.**

| Dimension | Wisconsin River (05397500) | Your gauge: ___________ |
|-----------|---------------------------|-------------------------|
| Mean discharge (m³/s) | ~80 | |
| Test NSE | ~0.70+ | |
| Test KGE | ~0.65+ | |
| Hydrograph shape | Strong April snowmelt peak | |
| Where model struggles most | Spring peak underestimation | |

---

**(i) Hydrograph shape** — describe what you see in the Q7c plot. Is there a dominant seasonal peak? Is the signal smooth or flashy? What physical property of your catchment explains this?

*Your answer:*

**(ii) Model accuracy** — is your NSE higher or lower than the Wisconsin River benchmark (~0.70)? Explain physically why your catchment type is easier or harder for the LSTM to model.

*Your answer:*

**(iii) Where the model struggles** — look at the error shading in Q7c. When is the shading thickest? What process is happening at that time of year, and why might the LSTM miss it?

*Your answer:*


---

## ✅ Submission Checklist

Before submitting, confirm:

- [ ] Setup cell ran without errors (all functions defined, Wisconsin River data downloaded)
- [ ] Q1–Q5 theory answers written in the Markdown cells
- [ ] Q6a baseline NSE/KGE/PBIAS printed
- [ ] Q6b all three `???` values filled in and modified model trained
- [ ] Q6b loss curve plot visible with both models
- [ ] Q6c comparison table filled in with real numbers, all three written answers complete
- [ ] Q7a `???` gauge ID filled in and data downloaded successfully
- [ ] Q7b new gauge trained and NSE/KGE/PBIAS printed
- [ ] Q7c both `???` array names filled in and hydrograph visible
- [ ] Q7d comparison table filled in, all three written answers reference actual numbers
- [ ] File saved as `lab05_eval_NETID.ipynb`

---

## 📊 Marks Summary

| Question | Topic | Marks |
|----------|-------|-------|
| Q1 | Sequential data & vanishing gradient | 3 |
| Q2 | LSTM gates & cell state | 3 |
| Q3 | Data prep, sliding window, chronological split | 3 |
| Q4 | Training dynamics | 2 |
| Q5 | NSE, KGE, PBIAS metrics | 2 |
| Q6a | Train baseline, record metrics | 1 |
| Q6b | Modify architecture, retrain, plot curves | 3 |
| Q6c | Comparison table + written interpretation | 2 |
| Q7a | Download new gauge, print statistics | 1 |
| Q7b | Preprocess + train on new gauge | 2 |
| Q7c | Plot test hydrograph | 1 |
| Q7d | Written comparison with physical reasoning | 2 |
| **Total** | | **25** |

---
> *"The best model is not the one with the highest NSE — it is the one whose errors you understand."*
